# MathE recommendations: live demo

A learner works on a MathE question. MathE sends the question to the Dataset Recommendation Service and gets back the most relevant teaching materials.

In [1]:
import os, statistics, sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from demo.mathe_api_demo import Mirror, base_url, get_token, recommend, relation

BASE_URL = base_url()
token = get_token()
mirror = Mirror()
pd.set_option("display.max_colwidth", 60)
print("Ready:", BASE_URL)

Ready: https://datagems-dev.scayle.es


## 1. The question and the request MathE sends

MathE sends the question ID, its content (LaTeX) and the maximum number of recommendations. The topic and keywords are MathE's own metadata for the question.

In [2]:
QUESTION_ID = "82"   # "Differentiate y = 2x sin(x)", the deliverable example
info = mirror.question_info(QUESTION_ID)
if not info:
    raise ValueError(f"Question {QUESTION_ID} is not in the MathE mirror; pick an existing question ID")
question = info["question"]   
payload = {"question_id": QUESTION_ID, "question": question, "n": 4}

pd.Series({
    "question_id": QUESTION_ID,
    "question": question,
    "n": payload["n"],
    "topic / subtopic": f"{info.get('topic', '')} / {info.get('subtopic', '')}",
    "keywords": ", ".join(info.get("keywords", [])),
}, name="request").to_frame()

,request
question_id,82
question,Differentiate $\displaystyle{y = 2x\sin(x)}$.
n,4
topic / subtopic,Differentiation / Derivatives
keywords,"Product rule, Trigonometric rules"


## 2. Document recommendations

In [3]:
def ask(content, payload):
    body, ms = recommend(BASE_URL, token, content, payload)
    ids = [str(r["material_id"]) for r in body["recommendations"]]
    details = mirror.materials(ids)
    rows = []
    for mid in ids:
        m = details.get(mid, {})
        why = relation(info, m) if m else {"same_topic": False, "shared_keywords": []}
        rows.append({
            "title": m.get("title", ""),
            "type": m.get("kind", ""),
            "same topic / subtopic": "yes" if why["same_topic"] else "no",
            "shared keywords": ", ".join(why["shared_keywords"]) or "— (content similarity)",
            "material_id": mid,
        })
    print(f"{content}: {len(ids)} recommendations in {ms:,.0f} ms")
    return pd.DataFrame(rows, index=pd.RangeIndex(1, len(rows) + 1, name="rank"))

ask("documents", payload)

documents: 4 recommendations in 522 ms


,title,type,same topic / subtopic,shared keywords,material_id
rank,,,,,
1,Notes on The Product Rule - Trigonometric Functions,DOCX,yes,"Product rule, Trigonometric rules",24
2,Derivative of the Product - Worked Example,DOCX,yes,Product rule,28
3,Derivative of the Product - Example,DOCX,yes,Product rule,27
4,Derivative of the product - solved exercice,DOCX,yes,Product rule,25


## 3. Video recommendations

In [4]:
ask("videos", payload)

videos: 4 recommendations in 302 ms


,title,type,same topic / subtopic,shared keywords,material_id
rank,,,,,
1,Applying the chain rule and product rule,video,yes,"Product rule, Trigonometric rules",783
2,Product Rule: Trigonometric Functions,video,yes,"Product rule, Trigonometric rules",488
3,How to find the differentiate e^ -x sin 3x chain rule,video,yes,"Product rule, Trigonometric rules",768
4,Making Maths Easy,video,yes,"Product rule, Trigonometric rules",625


## 4. How fast?

Rankings are computed offline after each MathE sync, so a request only reads the stored ranking.

In [5]:
times = [recommend(BASE_URL, token, "documents", payload)[1] for _ in range(5)]
print("5 calls:", " · ".join(f"{t:,.0f} ms" for t in times), f"→ median {statistics.median(times):,.0f} ms")

5 calls: 377 ms · 350 ms · 302 ms · 305 ms · 333 ms → median 333 ms


## 5. For comparison: computed online

The same question with slightly different text has no stored ranking, so the service scores it live: it embeds the question and ranks the topic's materials at request time.

In [6]:
variant = dict(payload, question=payload["question"] + r" \quad")
_, live_ms = recommend(BASE_URL, token, "documents", variant)
print(f"stored ranking: {statistics.median(times):,.0f} ms   ·   computed live: {live_ms:,.0f} ms")

stored ranking: 333 ms   ·   computed live: 3,314 ms
